# Curva de Aprendizaje: ¿Cuántos datos son suficientes?

**Propósito.** Determinar EMPÍRICAMENTE el tamaño de datos de entrenamiento ideal, en vez de fijarlo a dedo. Se entrena cada modelo representativo con tamaños crecientes de train (10k, 25k, 50k, 100k por clase) y se grafica el F1-Macro de validación frente al tamaño. Donde la curva se aplana, ahí hay "suficiente" dato.

**Protocolo idéntico** al benchmark: N-BaIoT, split Leave-One-Device-Out (LODO), QuantileTransformer, LoRA-FP16, best-checkpoint por F1 de validación, 46 features fijas.

**Clave metodológica:** solo varía el tamaño del conjunto de ENTRENAMIENTO (subconjuntos anidados del mismo pool). Val y test se mantienen FIJOS (mismos dispositivos LODO) para que la comparación sea limpia. El QuantileTransformer se reajusta sobre cada subconjunto de train (como ocurriría en un escenario real con solo esa cantidad de datos).

**Modelos representativos** (abarcan capacidad + familia):
- BERT-Mini (11M) — piso de capacidad, encoder.
- Qwen2.5-0.5B (0.5B) — decoder pequeño, finalista.
- Llama-3.2-1B (1.2B) — techo de capacidad FP16, decoder.

Fundamento: Goodfellow, Bengio, Courville, *Deep Learning* (2016) sobre capacidad y complejidad de muestra; Raschka, arXiv:1811.12808 sobre curvas de aprendizaje. Requiere licencias HF aceptadas para Llama-3.2-1B (gated).

## 1. Instalación, Importación y Autenticación

In [1]:
!pip install -q kagglehub scikit-learn pandas numpy torch matplotlib
!pip install --upgrade -q git+https://github.com/huggingface/transformers.git
!pip install --upgrade -q peft accelerate
!pip install --upgrade -q torchao   # transformers (git) exige torchao>=0.16; Colab trae 0.10 -> DEBE actualizarse
!pip install -q huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.6 MB/s eta 0:00:00


In [1]:
import kagglehub
import pandas as pd
import numpy as np
import os, time, gc, glob, random
from collections import defaultdict
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, matthews_corrcoef, precision_score, recall_score, f1_score
from sklearn.preprocessing import QuantileTransformer
from transformers import AutoModel, AutoConfig
from peft import LoraConfig, get_peft_model, TaskType
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Autenticacion exitosa en HuggingFace.")
except Exception as e:
    print("Advertencia: No se encontro HF_TOKEN (Llama-3.2-1B es gated).")

RANDOM_SEED = 42
def fijar_semillas(seed=RANDOM_SEED):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
fijar_semillas()

Autenticacion exitosa en HuggingFace.


## 2. Hiperparámetros, modelos representativos y tamaños de datos

In [2]:
# ==========================================
# EXPERIMENTO DE CURVA DE APRENDIZAJE
# ==========================================
NUM_CLASSES = 3
EPOCHS = 2
LEARNING_RATE = 1e-4
NOISE_LEVEL = 0.01
ACCUMULATION_STEPS = 1
BATCH_SIZE_TRAIN = 32
BATCH_SIZE_TEST = 128
LORA_R = 8; LORA_ALPHA = 32; LORA_TARGET_MODULES = "all-linear"

# Tamanos de train a probar (POR CLASE). El total es 3x este valor.
TRAIN_SIZES_POR_CLASE = [10000, 25000, 50000, 100000]

# Modelos representativos (capacidad creciente, 2 familias)
MODELOS_REPRESENTATIVOS = [
    #{"name": "BERT-Mini", "id": "google/bert_uncased_L-4_H-256_A-4"},
    #{"name": "Qwen2.5-0.5B", "id": "Qwen/Qwen2.5-0.5B-Instruct"},
    {"name": "Llama-3.2-1B", "id": "meta-llama/Llama-3.2-1B-Instruct"},
    #{"name": "ModernBERT-base", "id": "answerdotai/ModernBERT-base"},  # opcional (encoder moderno)
]

# Split LODO (identico a los otros notebooks)
CAP_POR_CLASE_EVAL = 3000
TEST_DEVICES = []; VAL_DEVICES = []
N_TEST_DEVICES = 2; N_VAL_DEVICES = 1

# Features fijas en 46 para este experimento (la curva es de DATOS, no de features)
caracteristicas = 46
caracteristicas_usar = [
    "MI_dir_L5_weight", "MI_dir_L5_mean", "MI_dir_L1_mean", "MI_dir_L1_variance", "MI_dir_L0.1_weight", "MI_dir_L0.1_mean", "MI_dir_L0.1_variance", "MI_dir_L0.01_weight", "MI_dir_L0.01_mean",
    "MI_dir_L0.01_variance", "H_L5_weight", "H_L5_mean", "H_L1_mean", "H_L1_variance", "H_L0.1_weight", "H_L0.1_mean", "H_L0.1_variance", "H_L0.01_weight", "H_L0.01_mean", "H_L0.01_variance",
    "HH_L5_mean", "HH_L5_magnitude", "HH_L1_weight", "HH_L0.1_weight", "HH_L0.1_mean", "HH_L0.01_weight", "HH_L0.01_mean", "HH_L0.01_radius", "HH_jit_L5_mean", "HH_jit_L1_weight", "HH_jit_L1_mean",
    "HH_jit_L0.1_weight", "HH_jit_L0.1_mean", "HH_jit_L0.01_weight", "HH_jit_L0.01_mean", "HpHp_L5_mean", "HpHp_L0.1_weight", "HpHp_L0.01_weight", "HpHp_L0.01_std", "H_L5_variance", "H_L3_variance",
    "HH_L3_weight", "HH_L0.1_covariance", "HH_L0.1_pcc", "HH_jit_L0.1_variance", "HpHp_L0.1_radius"
]
print(f"{len(caracteristicas_usar)} features | modelos: {[m['name'] for m in MODELOS_REPRESENTATIVOS]} | tamanos/clase: {TRAIN_SIZES_POR_CLASE}")

46 features | modelos: ['Llama-3.2-1B'] | tamanos/clase: [10000, 25000, 50000, 100000]


## 3. Arquitectura (idéntica al benchmark LoRA)

In [3]:
class SLMEdgeMulticlass(nn.Module):
    def __init__(self, checkpoint, num_features, num_classes=NUM_CLASSES):
        super().__init__()
        self.config = AutoConfig.from_pretrained(checkpoint, trust_remote_code=True)
        pad_tok = getattr(self.config, "pad_token_id", None)
        if isinstance(pad_tok, list):
            self.config.pad_token_id = pad_tok[0]
        elif pad_tok is None:
            eos_tok = getattr(self.config, "eos_token_id", 0)
            self.config.pad_token_id = eos_tok[0] if isinstance(eos_tok, list) else eos_tok
        self.transformer = AutoModel.from_pretrained(checkpoint, config=self.config, trust_remote_code=True, torch_dtype=torch.float32)

        #Agrege esta linea para la prueba
        self.transformer.gradient_checkpointing_enable()

        self.hidden_size = getattr(self.config, "hidden_size", getattr(self.config, "d_model", 768))
        self.embed_size = getattr(self.config, "embedding_size", self.hidden_size)
        self.feature_projector = nn.Linear(1, self.embed_size).to(torch.float32)
        nn.init.xavier_uniform_(self.feature_projector.weight)
        self.feature_embeddings = nn.Parameter(torch.randn(1, num_features, self.embed_size, dtype=torch.float32) * 0.02)
        self.classifier = nn.Sequential(nn.LayerNorm(self.hidden_size), nn.Linear(self.hidden_size, num_classes)).to(torch.float32)


    def forward(self, x, noise_level=0.0):
        x = x.to(self.feature_projector.weight.dtype)
        if self.training and noise_level > 0:
            x = x + torch.randn_like(x) * noise_level
        x = x.unsqueeze(-1)
        tokens = self.feature_projector(x) + self.feature_embeddings
        if getattr(self.config, "is_encoder_decoder", False):
            outputs = self.transformer.encoder(inputs_embeds=tokens)
        else:
            outputs = self.transformer(inputs_embeds=tokens)
        hidden_states = outputs.last_hidden_state if hasattr(outputs, 'last_hidden_state') else outputs[0]
        pooled = hidden_states.mean(dim=1)
        return self.classifier(pooled)

## 4. Pipeline LODO (devuelve arrays crudos) y utilidades

In [4]:
def _device_id(ruta):
    return os.path.basename(ruta).split('.')[0]

def _leer_muestreado(f, cap):
    total = sum(1 for _ in open(f, 'r')) - 1
    if total > cap:
        skip = sorted(np.random.choice(range(1, total + 1), total - cap, replace=False))
        return pd.read_csv(f, usecols=caracteristicas_usar, skiprows=skip).dropna()
    return pd.read_csv(f, usecols=caracteristicas_usar).dropna()

def preparar_lodo_crudo(path):
    """Devuelve arrays CRUDOS (sin escalar) de train/val/test bajo split LODO por
    dispositivo. El escalado se hace luego, por cada tamano de la curva."""
    clases = {0: '*.benign.csv', 1: '*.mirai.*.csv', 2: '*.gafgyt.*.csv'}
    dev_files = defaultdict(lambda: defaultdict(list))
    for label, patron in clases.items():
        archivos = sorted(glob.glob(os.path.join(path, '**', patron), recursive=True))
        if not archivos: archivos = sorted(glob.glob(os.path.join(path, patron)))
        for f in archivos:
            dev_files[_device_id(f)][label].append(f)
    dispositivos = sorted(dev_files.keys())
    completos = [d for d in dispositivos if all(len(dev_files[d][c]) > 0 for c in (0,1,2))]
    print(f"Dispositivos ({len(dispositivos)}): {dispositivos}")
    print(f"  Aptos val/test (3 clases): {completos}")
    print(f"  Solo train (sin alguna clase): {[d for d in dispositivos if d not in completos]}")
    if TEST_DEVICES or VAL_DEVICES:
        test_dev, val_dev = list(TEST_DEVICES), list(VAL_DEVICES)
    else:
        test_dev = completos[-N_TEST_DEVICES:]
        val_dev = completos[-(N_TEST_DEVICES + N_VAL_DEVICES):-N_TEST_DEVICES]
    train_dev = [d for d in dispositivos if d not in test_dev and d not in val_dev]
    for d in test_dev + val_dev:
        assert d in completos, f"{d} en val/test no tiene las 3 clases"
    print(f"  -> TRAIN:{train_dev}  VAL:{val_dev}  TEST:{test_dev}")

    def cargar(devs, cap_por_clase):
        dfs = []
        for d in devs:
            for label in (0,1,2):
                files = dev_files[d][label]
                if not files: continue
                cap_arch = max(1, cap_por_clase // (len(devs) * len(files)))
                for f in files:
                    try:
                        df = _leer_muestreado(f, cap_arch); df['label'] = label; dfs.append(df)
                    except Exception as e:
                        print(f"   [AVISO] {os.path.basename(f)} -> {e}")
        return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    # train con tope alto (el pool completo); val/test con tope de eval
    df_train = cargar(train_dev, max(TRAIN_SIZES_POR_CLASE))
    df_val   = cargar(val_dev,   CAP_POR_CLASE_EVAL)
    df_test  = cargar(test_dev,  CAP_POR_CLASE_EVAL)

    def balancear(df):
        n = df['label'].value_counts().min()
        return pd.concat([df[df.label==c].sample(n=n, random_state=RANDOM_SEED) for c in sorted(df.label.unique())], ignore_index=True)
    df_train, df_val, df_test = balancear(df_train), balancear(df_val), balancear(df_test)
    for nom, df in [('TRAIN(pool)',df_train),('VAL',df_val),('TEST',df_test)]:
        c = df['label'].value_counts().sort_index()
        print(f"   {nom}: Benign={c.get(0,0)} Mirai={c.get(1,0)} Bashlite={c.get(2,0)}")

    yt = df_train.pop('label').values; Xt = df_train.values
    yv = df_val.pop('label').values;   Xv = df_val.values
    yte = df_test.pop('label').values; Xte = df_test.values
    return Xt, yt, Xv, yv, Xte, yte

def subsample_balanceado(X, y, n_por_clase):
    """Subconjunto balanceado de n_por_clase muestras por clase (anidado, semilla fija)."""
    rng = np.random.RandomState(RANDOM_SEED)
    idx = []
    for c in np.unique(y):
        ci = np.where(y == c)[0]
        idx.extend(rng.choice(ci, min(n_por_clase, len(ci)), replace=False))
    idx = np.array(idx); rng.shuffle(idx)
    return X[idx], y[idx]

def capturar_metricas(model, loader, device):
    model.eval(); y_true, y_pred = [], []
    with torch.no_grad():
        for bx, by in loader:
            res = model(bx.to(device))
            y_pred.extend(torch.argmax(res, dim=1).cpu().numpy()); y_true.extend(by.numpy())
    return {"F1-Macro": f1_score(y_true, y_pred, average='macro', zero_division=0),
            "MCC": matthews_corrcoef(y_true, y_pred)}

def mk_loader(X, y, bs, sh):
    return DataLoader(TensorDataset(torch.tensor(X).float(), torch.tensor(y).long()),
                      batch_size=bs, shuffle=sh, pin_memory=True, num_workers=2)

def snapshot_entrenables(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def restaurar_entrenables(model, snap):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in snap: p.copy_(snap[n].to(p.device))

## 5. Bucle de la curva de aprendizaje (tamaño × modelo)

In [ ]:
path = kagglehub.dataset_download("mkashifn/nbaiot-dataset")
Xt_raw, yt, Xv_raw, yv, Xte_raw, yte = preparar_lodo_crudo(path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = (device.type == "cuda")
CSV_RESULTADOS = "curva_aprendizaje_46feat_1.csv"
historico = []

# Tamano exterior: se escala una vez por tamano y se reutiliza para todos los modelos
for size in TRAIN_SIZES_POR_CLASE:
    Xt_sub, yt_sub = subsample_balanceado(Xt_raw, yt, size)
    scaler = QuantileTransformer(output_distribution='normal', random_state=RANDOM_SEED)
    Xt_s = scaler.fit_transform(Xt_sub)          # fit SOLO en el subconjunto de train
    Xv_s = scaler.transform(Xv_raw); Xte_s = scaler.transform(Xte_raw)
    train_loader = mk_loader(Xt_s, yt_sub, BATCH_SIZE_TRAIN, True)
    val_loader   = mk_loader(Xv_s, yv, BATCH_SIZE_TEST, False)
    test_loader  = mk_loader(Xte_s, yte, BATCH_SIZE_TEST, False)
    print(f"\n===== TAMANO = {size}/clase ({len(yt_sub)} train total) =====")

    for slm in MODELOS_REPRESENTATIVOS:
        model = None; optimizer = None; best_snap = None
        try:
            nombre = slm['name']
            model = SLMEdgeMulticlass(slm['id'], len(caracteristicas_usar), num_classes=NUM_CLASSES).to(device)
            peft_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGET_MODULES, task_type=TaskType.FEATURE_EXTRACTION)
            model.transformer = get_peft_model(model.transformer, peft_config)
            optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
            criterion = nn.CrossEntropyLoss()
            scaler_amp = torch.amp.GradScaler('cuda', enabled=USE_AMP)

            best_val = -1.0; best_epoch = -1
            for epoch in range(1, EPOCHS + 1):
                model.train()
                for i, (bx, by) in enumerate(train_loader):
                    bx, by = bx.to(device), by.to(device)
                    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                        out = model(bx, noise_level=NOISE_LEVEL)
                        loss = criterion(out, by) / ACCUMULATION_STEPS
                    scaler_amp.scale(loss).backward()
                    if (i + 1) % ACCUMULATION_STEPS == 0:
                        scaler_amp.step(optimizer); scaler_amp.update(); optimizer.zero_grad()
                vq = capturar_metricas(model, val_loader, device)
                if vq['F1-Macro'] > best_val:
                    best_val = vq['F1-Macro']; best_snap = snapshot_entrenables(model); best_epoch = epoch
            if best_snap is not None: restaurar_entrenables(model, best_snap)

            del loss; torch.cuda.empty_cache(); gc.collect()
            model.half(); model.eval()
            qt = capturar_metricas(model, test_loader, device)
            fila = {"Modelo": nombre, "TrainSizePorClase": size, "TrainTotal": len(yt_sub),
                    "Val_F1": round(best_val, 4), "Test_F1": round(qt['F1-Macro'], 4),
                    "Test_MCC": round(qt['MCC'], 4), "Best_Epoch": best_epoch}
            pd.DataFrame([fila]).to_csv(CSV_RESULTADOS, mode="a", header=not os.path.exists(CSV_RESULTADOS), index=False)
            historico.append(fila)
            print(f"   {nombre:14s} | val F1={best_val:.4f} | test F1={qt['F1-Macro']:.4f} MCC={qt['MCC']:.4f}")
        except Exception as e:
            print(f"   Error en {slm['name']} @ {size}: {e}")
        finally:
            del model, optimizer, best_snap
            torch.cuda.empty_cache(); gc.collect(); time.sleep(1)

print(f"\nListo. Resultados en {CSV_RESULTADOS}")
display(pd.DataFrame(historico))

Using Colab cache for faster access to the 'nbaiot-dataset' dataset.
Dispositivos (9): ['1', '2', '3', '4', '5', '6', '7', '8', '9']
  Aptos val/test (3 clases): ['1', '2', '4', '5', '6', '8', '9']
  Solo train (sin alguna clase): ['3', '7']
  -> TRAIN:['1', '2', '3', '4', '5', '7']  VAL:['6']  TEST:['8', '9']
   TRAIN(pool): Benign=66660 Mirai=66660 Bashlite=66660
   VAL: Benign=3000 Mirai=3000 Bashlite=3000
   TEST: Benign=3000 Mirai=3000 Bashlite=3000

===== TAMANO = 10000/clase (30000 train total) =====


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


## 6. Gráfica: F1 vs tamaño de datos (¿dónde se aplana?)

In [ ]:
df = pd.read_csv(CSV_RESULTADOS)
plt.figure(figsize=(8, 5))
for nombre, g in df.groupby("Modelo"):
    g = g.sort_values("TrainSizePorClase")
    plt.plot(g["TrainSizePorClase"], g["Test_F1"], marker='o', label=nombre)
plt.xlabel("Muestras de entrenamiento por clase")
plt.ylabel("F1-Macro (test, LODO)")
plt.title("Curva de aprendizaje: F1 vs tamano de datos")
plt.grid(True, alpha=0.3); plt.legend()
plt.savefig("curva_aprendizaje.png", dpi=150, bbox_inches='tight')
plt.show()

print("Lectura: el tamano donde cada linea se aplana es 'suficiente' para ese modelo.")
print("Si las 3 lineas se aplanan al mismo tamano -> fija ese CAP_POR_CLASE_TRAIN para todo el benchmark.")
print("Si el modelo grande (Llama) sigue subiendo -> la capacidad exige mas datos; elige el maximo comun.")

FileNotFoundError: [Errno 2] No such file or directory: 'curva_aprendizaje_46feat.csv'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Diccionario de datos recolectados manualmente
datos_recolectados = [
    # Resultados finalizados de BERT-Mini
    {"Modelo": "BERT-Mini", "TrainSizePorClase": 10000,  "Test_F1": 0.9853},
    {"Modelo": "BERT-Mini", "TrainSizePorClase": 25000,  "Test_F1": 0.9988},
    {"Modelo": "BERT-Mini", "TrainSizePorClase": 50000,  "Test_F1": 0.9946},
    {"Modelo": "BERT-Mini", "TrainSizePorClase": 100000, "Test_F1": 0.9997},

    # Descomenta y actualiza estas líneas cuando termine Qwen2.5-0.5B
    # {"Modelo": "Qwen2.5-0.5B", "TrainSizePorClase": 10000,  "Test_F1": 0.0000},
    # {"Modelo": "Qwen2.5-0.5B", "TrainSizePorClase": 25000,  "Test_F1": 0.0000},
    # {"Modelo": "Qwen2.5-0.5B", "TrainSizePorClase": 50000,  "Test_F1": 0.0000},
    # {"Modelo": "Qwen2.5-0.5B", "TrainSizePorClase": 100000, "Test_F1": 0.0000},

    # Descomenta y actualiza estas líneas cuando termine Llama-3.2-1B
    # {"Modelo": "Llama-3.2-1B", "TrainSizePorClase": 10000,  "Test_F1": 0.0000},
    # {"Modelo": "Llama-3.2-1B", "TrainSizePorClase": 25000,  "Test_F1": 0.0000},
    # {"Modelo": "Llama-3.2-1B", "TrainSizePorClase": 50000,  "Test_F1": 0.0000},
    # {"Modelo": "Llama-3.2-1B", "TrainSizePorClase": 100000, "Test_F1": 0.0000},
]

# 2. Convertir a DataFrame en lugar de leer el CSV
df = pd.DataFrame(datos_recolectados)

# 3. Lógica original de graficado
plt.figure(figsize=(8, 5))
for nombre, g in df.groupby("Modelo"):
    g = g.sort_values("TrainSizePorClase")
    plt.plot(g["TrainSizePorClase"], g["Test_F1"], marker='o', label=nombre)

plt.xlabel("Muestras de entrenamiento por clase")
plt.ylabel("F1-Macro (test, LODO)")
plt.title("Curva de aprendizaje: F1 vs tamaño de datos")
plt.grid(True, alpha=0.3)
plt.legend()

plt.savefig("curva_aprendizaje.png", dpi=150, bbox_inches='tight')
plt.show()

print("Lectura: el tamaño donde cada línea se aplana es 'suficiente' para ese modelo.")
print("Si las 3 líneas se aplanan al mismo tamaño -> fija ese CAP_POR_CLASE_TRAIN para todo el benchmark.")
print("Si el modelo grande (Llama) sigue subiendo -> la capacidad exige más datos; elige el máximo común.")